# Othello — evidence-rule benchmark

The same five matched arms as the Connect 4 benchmark, on a game with **65
actions** (64 squares plus an explicit pass) and **~60-ply** games, against
Connect 4's 7 actions and ~34 plies.

The question: **MA (`search=additive_mle`, `target=additive`) came out on top in
Connect 4 — does that survive a branching factor an order of magnitude larger?**
In Connect 4 the four ThompsonZero arms were statistically indistinguishable
(every pairwise p > 0.45), so "MA best" was a ranking, not a separation. A bigger
game gives the rules more room to differ.

| | `target = additive` | `target = additive_mle` |
|---|---|---|
| **`search = additive`** | `AA` | `AM` |
| **`search = additive_mle`** | `MA` | `MM` |

plus `AZ`, an ordinary AlphaZero control (policy + scalar value, PUCT,
visit-count targets) held to the same trunk, self-play shape, optimiser, LR
schedule, batch size and MCTS-Solver overlay.

**No solved-position metric here.** That one needs an exact oracle and there is
no Othello equivalent of Pons' Connect 4 test sets, so strength is measured by
the relative ladder and the round robin only.

In [ ]:
%pip install open_spiel -q
import os, sys, urllib.request
_BRANCH = 'claude/connect4-dirichlet-values-my96dt'
_BASE = ('https://raw.githubusercontent.com/calvinpozderac-claude/open_spiel/'
         f'{_BRANCH}/open_spiel/colabs/')
for _f in ('connect4_dirichlet_utils.py', 'connect4_alphazero_utils.py',
           'connect4_solved_eval.py', 'connect4_benchmark.py',
           'connect4_benchmark_tests.py', 'connect4_dirichlet_tests.py',
           'othello_benchmark.py', 'othello_benchmark_tests.py'):
    _p = next((p for p in (_f, os.path.join('open_spiel', 'colabs', _f),
                           os.path.join('..', 'colabs', _f))
               if os.path.exists(p)), None)
    if _p is None:
        urllib.request.urlretrieve(_BASE + _f, _f); _p = _f
    _d = os.path.dirname(os.path.abspath(_p))
    if _d not in sys.path:
        sys.path.insert(0, _d)

import importlib
import connect4_dirichlet_utils as c4
import connect4_alphazero_utils as az
import connect4_benchmark as _cb
import othello_benchmark as bench
for _m in (c4, az, _cb, bench):
    importlib.reload(_m)
print('loaded', bench.__file__)

## The game, and the settings it forces

In [ ]:
# Othello is not Connect 4 with a bigger board: ~8.5 legal moves per position
# against ~6, ~60 plies against ~34, and an explicit pass action.  Connect 4's
# max_plies=42 would truncate most games and score them as draws.
bench.describe()

In [ ]:
shared = bench.default_shared(
    root         = 'othello_benchmark',
    num_episodes = 4000,
    seed         = 0,
    device       = 'auto',
    # Everything else follows othello_benchmark.OTHELLO_DEFAULTS; see the module
    # docstring for why each one differs from Connect 4.
)
for k in ('game', 'max_plies', 'temp_threshold', 'fast_sims', 'full_sims',
          'root_noise_alpha', 'eval_sims'):
    print(f'{k:18} {shared[k]}')

### Are the arms actually matched?

The trunk is identical by construction. The heads are not, and at 65 actions
that stops being a rounding error — ThompsonZero emits 4 numbers per action
(260) where AlphaZero emits one logit plus a value (66).

In [ ]:
bench.report_params(shared)

### How big should the network be?

`32/3/8` was carried over from Connect 4 — a solved ~10^13-state game. Othello is
~10^28 with 60-ply games, and 8x8 board games at this scale are normally run at
**64-128 channels and 5-10 residual blocks** (roughly 0.5-3M parameters), not
195k.

The table below measures parameters and forward cost per candidate trunk, and
anchors the hours to the 0.65 games/s the Connect 4 runs actually achieved on an
RX 5700 XT. Two things fall out of it:

- Othello costs **~3.5x more NN evaluations per game** than Connect 4 (60 plies x
  150 sims vs 34 x 75), on a 1.5x bigger board. Even the *tiny* trunk is ~46 h
  for 5 arms at 4000 episodes.
- **The TZ/AZ parameter ratio falls as the trunk grows** (2.09x at 32/3, 1.24x at
  64/5, 1.08x at 128/8), because the head cost is fixed by the action count while
  the trunk scales. Most of the "mismatch" is an artifact of an undersized trunk.

In [ ]:
bench.sizing_table(shared, n_arms=5)

### Equalising capacity

Two definitions of "matched" and they cannot both hold:

- **identical trunks** (the default) — ThompsonZero ends up with more total
  parameters, because it emits 4 numbers per action against AlphaZero's one.
- **equal totals** (`match_capacity`) — AlphaZero gets a wider trunk to
  compensate, so it ends up with *more* of the capacity that actually computes.

ThompsonZero's extra parameters are an output projection it needs to emit 260
numbers instead of 66; that is a requirement, not an advantage. So identical
trunks is the more defensible default, and `match_capacity` over-corrects. Use it
if you want the totals equal and are willing to accept the other bias — and note
that at a trunk size suited to Othello there is little left to correct either way.

In [ ]:
# Uncomment to equalise total parameters instead of trunks.
# shared = bench.match_capacity(shared)
# bench.report_params(shared)

## Train

In [ ]:
# Sequential on purpose: each arm already saturates the GPU through its own
# batched inference server, so running them concurrently would only distort the
# perf numbers.  Resumable — re-run the cell and each arm picks up its latest.pt.
hists = bench.train_all(shared)

## Round robins

In [ ]:
# SEARCH-FREE (raw network quality), all generations on one scale.
players = bench.load_players(shared, gens=(1000, 2000, 4000))
names, W, elo = bench.round_robin(players, sims=0, games_per_pair=60)
bench.report(names, W, elo, 'search-free · all generations')

In [ ]:
# WITH SEARCH, final checkpoints only — the deployed configuration.
finals = bench.load_players(shared, gens=(4000,))
names_m, W_m, elo_m = bench.round_robin(finals, sims=128, games_per_pair=40)
bench.report(names_m, W_m, elo_m, 'MCTS-128 · final checkpoints')
bench.head_to_head_table(names_m, W_m)

### Does the Connect 4 ranking hold?

In Connect 4 the four ThompsonZero arms were statistically indistinguishable.
Check the same thing here before reading anything into the order: with 40 games
per pair the standard error on a pair score is about 8%, so a gap under ~16% is
noise.

In [ ]:
import numpy as np
tz = [n for n in names_m if not n.startswith(('AZ', 'random'))]
print(f'{"pair":<20}{"score":>9}{"games":>8}{"95% CI":>18}')
for i, a in enumerate(tz):
    for b in tz[i + 1:]:
        ia, ib = names_m.index(a), names_m.index(b)
        n = W_m[ia][ib] + W_m[ib][ia]
        if n == 0:
            continue
        s = W_m[ia][ib] / n
        lo, hi = bench.wilson(s, n)
        flag = '' if lo < 0.5 < hi else '   <-- separated'
        print(f'{a+" vs "+b:<20}{s:>8.1%}{n:>8.0f}   [{lo:.1%}, {hi:.1%}]{flag}')

## Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for name, h in hists.items():
    if not h.get('ep'):
        continue
    ax[0].plot(h['ep'], h['loss'], label=name)
    ax[1].plot(h['ep'], h['draw_pct'], label=name)
    ax[2].plot(h['ep'], h['plies'], label=name)
for a, t in zip(ax, ('total loss', 'draw %', 'game length (plies)')):
    a.set_title(t); a.set_xlabel('episode'); a.legend()
plt.tight_layout(); plt.show()

# The deep-eval Elo ladder, now reported by BOTH engines.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for name, h in hists.items():
    if not h.get('elo'):
        continue
    eps = [e for e, _ in zip(h['ep'], h['elo'])]
    ax.plot(eps[:len(h['elo'])],
            [d.get(str(e), np.nan) for e, d in zip(eps, h['elo'])],
            marker='o', label=name)
ax.set_xlabel('episode'); ax.set_ylabel(f'Elo @ MCTS-{shared["eval_sims"]}')
ax.set_title('deep-eval ladder'); ax.legend()
plt.tight_layout(); plt.show()

## Is the AlphaZero control sound?

It lost Connect 4 by ~450 Elo, which was partly two real bugs (root noise reached
only the first move; no solver-labelled training samples) and partly an unfair
search-free comparison. All three are fixed. These diagnostics separate "the
method is behind" from "the control is broken": if search adds far more for AZ
than for the ThompsonZero arms, the AZ *network* is the weak part, not its
search.

In [ ]:
bench.search_value(shared, arms=('MA', 'AZ'), gen=4000, sims=128, games=40)
bench.az_progression(shared, gens=(1000, 2000, 4000), sims=128, games=40)
bench.az_cpuct_sweep(shared, gen=4000, opponent='MA', sims=128, games=60)

## Self-tests

In [ ]:
import subprocess
_d = os.path.dirname(bench.__file__)
for _t in ('connect4_dirichlet_tests.py', 'connect4_benchmark_tests.py',
           'othello_benchmark_tests.py'):
    _r = subprocess.run([sys.executable, os.path.join(_d, _t)], cwd=_d,
                        capture_output=True, text=True)
    print(_t, '->', _r.stdout.strip().splitlines()[-1] if _r.stdout
          else _r.stderr[-300:])